# Eclat demo on Google Colab

The same steps run on a local Julia kernel in `demo_local.ipynb`, which differs only in setup.

Eclat (Zaki, IEEE TKDE 2000) finds every itemset that occurs in at least a minsup fraction of the transactions. It keeps the tidset of each itemset, the set of transactions that contain it, and gets the support of a longer itemset by intersecting the tidsets of two shorter ones. This notebook runs the team's Julia implementation step by step: first on a four-basket file worked out by hand, then on the Mushroom benchmark, where both variants are compared with an independent SPMF run.

Run all cells on a CPU runtime. Colab has no Julia kernel, so two Python cells install Julia 1.13.2 and Java 25 and check each download against a pinned SHA-256. Every later step is a `%%shell` cell that holds the same commands you would type in a terminal from the repository root. `set -euo pipefail` and Colab stop the notebook at the first command that fails. The Python cells after them only read, check and display what the commands wrote. To replay a run, set `ITEMSET_REVISION` in the first code cell to a full commit SHA.

The two applications run in their own notebooks, [market_basket_colab.ipynb](https://colab.research.google.com/github/NgTHung/Itemset/blob/main/notebooks/market_basket_colab.ipynb) and [image_patterns_colab.ipynb](https://colab.research.google.com/github/NgTHung/Itemset/blob/main/notebooks/image_patterns_colab.ipynb).

In [ ]:
import hashlib
import os
import re
import subprocess
import tarfile
import urllib.request
from pathlib import Path

REPOSITORY = "https://github.com/NgTHung/Itemset"
ITEMSET_REVISION = ""  # A full commit SHA replays a run; empty takes the head of the default branch.
JULIA_URL = "https://julialang-s3.julialang.org/bin/linux/x64/1.13/julia-1.13.2-linux-x86_64.tar.gz"
JULIA_SHA256 = "2ecdfc9890ab0a565fdd92404c915ba2a04a933dbd5d292cc224d13819d17d07"

root, julia_home = Path("/content/Itemset"), Path("/content/julia-1.13.2")
revision = ITEMSET_REVISION or os.environ.get("ITEMSET_REVISION") or subprocess.check_output(
    ["git", "ls-remote", "--exit-code", REPOSITORY, "HEAD"], text=True, timeout=60).split()[0]
if not re.fullmatch(r"[0-9a-f]{40}", revision):
    raise ValueError(f"ITEMSET_REVISION must be a full lowercase 40-character commit SHA, got {revision!r}")
if not root.exists():
    subprocess.run(["git", "clone", "--", REPOSITORY, str(root)], check=True)
subprocess.run(["git", "-C", str(root), "checkout", "--detach", revision], check=True)

if not julia_home.exists():
    archive, digest = Path("/content/julia-1.13.2.tar.gz"), hashlib.sha256()
    with urllib.request.urlopen(JULIA_URL, timeout=60) as source, archive.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != JULIA_SHA256:
        archive.unlink()
        raise ValueError(f"Julia archive SHA-256 is {digest.hexdigest()}, expected {JULIA_SHA256}")
    julia_home.mkdir()
    with tarfile.open(archive) as package:
        package.extractall(julia_home, filter="data")
os.environ["PATH"] = f"{next(julia_home.glob('julia-*/bin'))}{os.pathsep}{os.environ['PATH']}"
os.environ["JULIA_NUM_THREADS"] = "1"
os.chdir(root)


def julia(*args):
    """Run Julia in the repository and stop the notebook on a nonzero exit, which a shell escape would not."""
    process = subprocess.Popen(["julia", "--startup-file=no", "--project=.", *args], stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait():
        raise RuntimeError(f"julia {' '.join(args)} exited with {process.returncode}")


print("Revision:", revision)
print(subprocess.check_output(["julia", "--version"], text=True).strip())

In [ ]:
import tomllib

JAVA_URL = "https://github.com/adoptium/temurin25-binaries/releases/download/jdk-25.0.5%2B8/OpenJDK25U-jre_x64_linux_hotspot_25.0.5_8.tar.gz"
JAVA_SHA256 = "6fac98f0c8202efc3135a9e9e42aa61963ec42d1c5cbb40caf231e7b14a24ef6"


def download_verified(url, path, expected):
    """Download url to path and delete the file unless its SHA-256 is the pinned one."""
    digest = hashlib.sha256()
    with urllib.request.urlopen(url, timeout=60) as source, path.open("wb") as target:
        while chunk := source.read(1024 * 1024):
            target.write(chunk)
            digest.update(chunk)
    if digest.hexdigest() != expected:
        path.unlink()
        raise ValueError(f"{path.name} SHA-256 is {digest.hexdigest()}, expected {expected}")


java_home = Path("/content/java-25")
if not java_home.exists():
    archive = Path("/content/java-25.tar.gz")
    download_verified(JAVA_URL, archive, JAVA_SHA256)
    java_home.mkdir()
    with tarfile.open(archive) as package:
        package.extractall(java_home, filter="data")
# Colab ships an older Java, and the pinned SPMF release needs Java 25, so the pinned one goes first on PATH.
os.environ["PATH"] = f"{next(java_home.glob('*/bin'))}{os.pathsep}{os.environ['PATH']}"

with (root / "tools/spmf/release.toml").open("rb") as stream:
    release = tomllib.load(stream)
jar = root / "tools/spmf/spmf.jar"
if not jar.exists():
    download_verified(release["url"], jar, release["sha256"])
elif hashlib.sha256(jar.read_bytes()).hexdigest() != release["sha256"]:
    raise ValueError(f"{jar} differs from the SPMF release pinned in tools/spmf/release.toml")
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.strip())
print(f"SPMF {release['version']}, SHA-256 {release['sha256']}")

In [ ]:
%%shell
set -euo pipefail
julia --startup-file=no --project=. -e 'using Pkg; Pkg.instantiate()'

In [ ]:
%%shell
set -euo pipefail
# The suite checks both variants against every hand-derived toy fixture and a brute-force count.
julia --startup-file=no --project=. test/runtests.jl

## A worked example

The input uses the SPMF format: one transaction per line, items as positive integers separated by spaces. `demo.jl` writes each frequent itemset with its support count as `items #SUP: count`. At minsup 0.5 an itemset must occur in 2 of the 4 baskets.

The expected file was worked out by hand. `tools/spmf/compare.jl` then compares the two complete maps, every itemset and every support, not only their counts.

In [ ]:
%%shell
set -euo pipefail
echo "Input, data/toy/baskets.txt:"
cat data/toy/baskets.txt
echo
mkdir -p results/raw/demo-toy
julia --startup-file=no --project=. demo.jl --minsup 0.5 --output results/raw/demo-toy/baskets.txt data/toy/baskets.txt

paste <(echo "mined by demo.jl"; cat results/raw/demo-toy/baskets.txt) \
      <(echo "worked by hand"; cat data/toy/baskets.expected.txt) | expand -t 24
echo
julia --startup-file=no --project=. tools/spmf/compare.jl data/toy/baskets.expected.txt results/raw/demo-toy/baskets.txt

## Mushroom

Mushroom is a dense benchmark from the FIMI repository: 8,124 transactions over 119 items, 23 items each. The next cell downloads it, checks its SHA-256 against `data/benchmark/manifest.toml`, and prints its statistics. The next cell mines it at minsup 0.3 with both variants, and the cell after it checks that the two outputs agree and shows the longest frequent itemsets.

In [ ]:
%%shell
set -euo pipefail
julia --startup-file=no --project=. experiments/prepare_benchmarks.jl mushroom

In [ ]:
%%shell
set -euo pipefail
julia --startup-file=no --project=. demo.jl --check-input data/benchmark/mushroom.txt
mkdir -p results/raw/demo-mushroom-mine
for variant in baseline optimized; do
  julia --startup-file=no --project=. demo.jl --minsup 0.3 --variant "$variant" \
    --output "results/raw/demo-mushroom-mine/$variant.txt" data/benchmark/mushroom.txt
done

In [ ]:
import pandas as pd
from IPython.display import display


def read_itemsets(path):
    """One row per output line `items #SUP: count`."""
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        items, support = line.split(" #SUP: ")
        rows.append({"items": items, "length": len(items.split()), "support": int(support)})
    return pd.DataFrame(rows)


outputs = {variant: read_itemsets(root / f"results/raw/demo-mushroom-mine/{variant}.txt")
           for variant in ("baseline", "optimized")}
for variant, table in outputs.items():
    print(f"{variant}: {len(table)} frequent itemsets at minsup 0.3")
same = {tuple(row) for row in outputs["baseline"].itertuples(index=False)} == \
       {tuple(row) for row in outputs["optimized"].itertuples(index=False)}
print("Both variants return the same itemsets and supports:", same)
if not same:
    raise RuntimeError("The baseline and optimized outputs differ; compare them with tools/spmf/compare.jl")
display(outputs["optimized"].sort_values(["length", "support"], ascending=False).head(10).reset_index(drop=True))

## Comparison with SPMF

`experiments/run.jl` runs baseline, optimized and SPMF Eclat on Mushroom at five thresholds from `experiments/configs/demo-mushroom.toml`. Each time covers one process from spawn to exit, including Julia or JVM startup. After timing, the harness compares every output with a separate SPMF Eclat run as complete maps; `exact` with match rate 1.0 means every itemset and every support agree.

One cold run per point shows the trend in a few minutes. The report's runtime figures use five repeats on a fixed host.

In [ ]:
%%shell
set -euo pipefail
# run.jl refuses a non-empty output folder, so a rerun of this cell starts from an empty one.
rm -rf results/raw/demo-mushroom
julia --startup-file=no --project=. experiments/run.jl experiments/configs/demo-mushroom.toml \
  --out results/raw/demo-mushroom --java "$(command -v java)"

In [ ]:
import pandas as pd
from IPython.display import display

sweep = root / "results/raw/demo-mushroom"
runs = pd.read_csv(sweep / "runs.csv")
# SPMF attempts add a second row with SPMF's own timer; keep the spawn-to-exit row of every variant.
runs = runs[runs["timing_scope"] == "process"]
# run.jl records a failed attempt as a row and still exits 0, so a failure has to stop the notebook here.
failed = runs[(runs["status"] != "ok") | (runs["correctness_status"] != "exact")]
if len(failed):
    print(failed[["attempt_id", "status", "correctness_status", "error"]].to_string(index=False))
    raise RuntimeError(f"{len(failed)} sweep attempts failed or differ from SPMF; see {sweep}/attempts")
display(runs[["minsup_relative", "variant", "output_count", "status", "correctness_status", "match_rate",
              "elapsed_seconds"]].sort_values(["minsup_relative", "variant"], ascending=[False, True])
        .reset_index(drop=True))

In [ ]:
import matplotlib.pyplot as plt

runs = pd.read_csv(root / "results/raw/demo-mushroom/runs.csv")
runs = runs[runs["timing_scope"] == "process"]
figure, (count_axis, time_axis) = plt.subplots(1, 2, figsize=(11, 4))
for variant, rows in runs.groupby("variant"):
    rows = rows.sort_values("minsup_relative")
    count_axis.plot(rows["minsup_relative"], rows["output_count"], marker="o", label=variant)
    time_axis.plot(rows["minsup_relative"], rows["elapsed_seconds"], marker="o", label=variant)
count_axis.set(yscale="log", xlabel="minsup (fraction of transactions)", ylabel="frequent itemsets",
               title="Output size on Mushroom")
time_axis.set(xlabel="minsup (fraction of transactions)", ylabel="process time (s)",
              title="Runtime on Mushroom, one cold run")
for axis in (count_axis, time_axis):
    axis.invert_xaxis()
    axis.grid(alpha=0.3)
    axis.legend()
figure.tight_layout()
plt.show()

The three count lines coincide because the outputs are equal. The x axes run from high to low minsup, the direction in which the output grows.

In [ ]:
import json
import zipfile

# Files go in by relative path, so the ZIP holds the same names whatever the runtime.
sweep = root / "results/raw/demo-mushroom"
files = sorted([sweep / "runs.csv", sweep / "config.toml", sweep / "environment.toml",
                *sweep.glob("attempts/*/comparison.toml"), root / "results/raw/demo-toy/baskets.txt"])
record = {
    "repository": REPOSITORY,
    "revision": revision,
    "julia": subprocess.check_output(["julia", "--version"], text=True).strip(),
    "julia_archive_sha256": JULIA_SHA256,
    "java": subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0],
    "java_archive_sha256": JAVA_SHA256,
    "spmf_sha256": release["sha256"],
    "sha256": {p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in files},
}
evidence = Path("/content/itemset-demo-evidence.zip")
with zipfile.ZipFile(evidence, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.relative_to(root).as_posix())
    archive.writestr("result.json", json.dumps(record, indent=2))
print("Evidence:", evidence, "with", len(files), "files and result.json")